# Calibration Phases 1-2 (Manual)

Baseline R0 calibration first, then Reff(t) calibration with mobility-driven multipliers.

In [ ]:
import sys
from pathlib import Path
from datetime import date
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    get_default_shp_region_config,
    is_openabm_available,
    run_baseline_r0_calibration_scenario,
    run_reff_calibration_scenario,
    InterventionSet,
    compile_network_multipliers,
)


## Controls (Edit First)

In [ ]:
# Core region/window controls
shp_name = "Helsinki and Uusimaa"
reference_start_date = "2020-03-01"
end_date = "2020-06-30"

# Explicit calibration controls requested
R0_target = 2.3
initial_infected = 100
simulated_population = 200000

# Phase 1 baseline grid
infectious_rate_grid = [4.0, 5.0, 5.8, 6.5, 7.0]
phase1_simulation_steps = 90  # shorter on purpose for speed
phase1_burn_in_days = 14      # ignore earliest stochastic days for R0 fit
phase1_eval_window_days = 21  # evaluate early-R on next valid window

# Phase 2: three controls (one per network in OpenABM runtime layer)
generation_interval = 5
smoothing_window = 7
Mhome_grid = [0.8, 1.0, 1.2]
Moccupation_grid = [0.5, 1.0, 1.5]
Mrandom_grid = [0.5, 1.0, 1.5]

# Occupation split in adapter (work = 1 - school)
occupation_school_weight = 0.3
occupation_work_weight = 1.0 - occupation_school_weight

# Runtime optimization controls
use_openabm = is_openabm_available()
quick_coarse_mode = True               # coarse pass with fewer combinations
coarse_use_openabm = False             # fast coarse ranking
refine_top_n = 3                       # run top-N combinations with OpenABM

# Paths
observed_cases_path = repo_root / "data" / "processed" / "thl_cases_2020_2022_processed_daily.csv"
timeline_processed_path = repo_root / "data" / "processed" / "oxcgrt_finland_2020_2022_timeline.csv"
mobility_processed_path = repo_root / "data" / "processed" / "google_mobility_hus_2020_2022.csv"

print("SHP:", shp_name)
print("Window:", reference_start_date, "->", end_date)
print("R0_target:", R0_target)
print("initial_infected:", initial_infected)
print("simulated_population:", simulated_population)
print("phase1_burn_in_days:", phase1_burn_in_days)
print("phase1_eval_window_days:", phase1_eval_window_days)
print("OpenABM available:", use_openabm)
print("coarse_use_openabm:", coarse_use_openabm)
print("refine_top_n:", refine_top_n)
print("M grids:", {"home": Mhome_grid, "occupation": Moccupation_grid, "random": Mrandom_grid})
print("occupation split:", {"work": occupation_work_weight, "school": occupation_school_weight})


## Region Setup

In [ ]:
region_config = get_default_shp_region_config(shp_name, simulated_population=simulated_population)

simulation_steps = (date.fromisoformat(end_date) - date.fromisoformat(reference_start_date)).days + 1
print("Resolved region:", region_config.name)
print("Real population:", region_config.real_population)
print("Simulated population:", region_config.simulated_population)
print("Phase2 simulation_steps:", simulation_steps)


## Phase 1: Baseline R0 Calibration (No Interventions)

In [ ]:
phase1_rows = []
phase1_runs = {}

for infectious_rate in infectious_rate_grid:
    run = run_baseline_r0_calibration_scenario(
        region_config=region_config,
        simulation_steps=phase1_simulation_steps,
        initial_infected=initial_infected,
        infectious_rate=infectious_rate,
        generation_interval=generation_interval,
        smoothing_window=smoothing_window,
        use_openabm=use_openabm,
    )
    phase1_runs[infectious_rate] = run

    ts = run["r_agent_smoothed"]
    series = pd.DataFrame({
        "t": pd.Series(ts.times, dtype=float),
        "r": pd.Series(ts.values, dtype=float),
    })
    series = series[(series["t"] >= float(phase1_burn_in_days)) & (~series["r"].isna())]
    eval_slice = series.head(phase1_eval_window_days)
    early_r = float(eval_slice["r"].mean()) if not eval_slice.empty else float("nan")

    if np.isnan(early_r):
        note = "no valid R"
    elif early_r < R0_target * 0.9:
        note = "below target"
    elif early_r > R0_target * 1.1:
        note = "above target"
    else:
        note = "near target"

    phase1_rows.append({
        "infectious_rate": infectious_rate,
        "initial_infected": initial_infected,
        "simulated_population": simulated_population,
        "phase1_burn_in_days": phase1_burn_in_days,
        "phase1_eval_window_days": phase1_eval_window_days,
        "early_r_level": early_r,
        "note": note,
    })

phase1_df = pd.DataFrame(phase1_rows).sort_values("infectious_rate").reset_index(drop=True)
phase1_df


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for infectious_rate, run in phase1_runs.items():
    ts = run["r_agent_smoothed"]
    ax.plot(ts.times, ts.values, label=f"infectious_rate={infectious_rate}", linewidth=2)

ax.axhline(R0_target, color="tab:gray", linestyle="--", linewidth=1.8, label=f"R0_target={R0_target}")
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.3, label="R=1")
ax.set_title("Phase 1: Smoothed R_agent by infectious_rate")
ax.set_xlabel("Simulation day")
ax.set_ylabel("R_agent (smoothed)")
ax.grid(alpha=0.3)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
phase1_df["abs_error_to_R0_target"] = (phase1_df["early_r_level"] - R0_target).abs()
best_phase1 = phase1_df.sort_values("abs_error_to_R0_target").iloc[0]
selected_infectious_rate = float(best_phase1["infectious_rate"])

print("Phase 1 pick (burn-in corrected):")
print(" selected_infectious_rate:", selected_infectious_rate)
print(" early_r_level:", float(best_phase1["early_r_level"]))
print(" burn_in_days:", int(best_phase1["phase1_burn_in_days"]))
print(" eval_window_days:", int(best_phase1["phase1_eval_window_days"]))
print(" note:", best_phase1["note"])


## Mobility Input (Create Local Placeholder If Missing)

In [ ]:
if not mobility_processed_path.exists():
    print("Mobility file not found. Creating a simple local placeholder for manual calibration workflow:", mobility_processed_path)
    dates = pd.date_range(start=reference_start_date, end=end_date, freq="D")

    # Simple first-wave proxy profile: strong drop in spring, partial recovery by summer.
    workplace = []
    retail = []
    for d in dates:
        if d < pd.Timestamp("2020-03-16"):
            workplace.append(-5.0)
            retail.append(-8.0)
        elif d <= pd.Timestamp("2020-05-31"):
            workplace.append(-35.0)
            retail.append(-45.0)
        else:
            workplace.append(-18.0)
            retail.append(-22.0)

    mobility_df = pd.DataFrame({
        "date": dates.date,
        "region": region_config.name,
        "workplace_percent_change_from_baseline": workplace,
        "retail_and_recreation_percent_change_from_baseline": retail,
    })
    mobility_processed_path.parent.mkdir(parents=True, exist_ok=True)
    mobility_df.to_csv(mobility_processed_path, index=False)

mobility_preview = pd.read_csv(mobility_processed_path)
print("Mobility rows:", len(mobility_preview))
display(mobility_preview.head())


## Phase 2: Reff(t) Calibration With Mobility

In [ ]:
all_triples = list(itertools.product(Mhome_grid, Moccupation_grid, Mrandom_grid))
if quick_coarse_mode:
    all_triples = [
        (1.0, 0.5, 0.5),
        (1.0, 1.0, 1.0),
        (1.0, 1.5, 1.5),
        (0.8, 1.0, 1.0),
        (1.2, 1.0, 1.0),
    ]

coarse_rows = []
coarse_runs = {}
for m_home, m_occupation, m_random in all_triples:
    weights = {
        "household": float(m_home),
        "work": float(m_occupation),
        "school": float(m_occupation),
        "random": float(m_random),
    }
    run = run_reff_calibration_scenario(
        region_config=region_config,
        observed_cases_path=str(observed_cases_path),
        timeline_processed_path=str(timeline_processed_path),
        mobility_processed_path=str(mobility_processed_path),
        reference_start_date=reference_start_date,
        end_date=end_date,
        initial_infected=initial_infected,
        simulated_population=simulated_population,
        infectious_rate=selected_infectious_rate,
        household_mobility_scale=m_home,
        work_mobility_scale=m_occupation,
        school_mobility_scale=m_occupation,
        random_mobility_scale=m_random,
        network_weights=weights,
        occupation_school_weight=occupation_school_weight,
        generation_interval=generation_interval,
        smoothing_window=smoothing_window,
        use_openabm=coarse_use_openabm,
    )
    coarse_runs[(m_home, m_occupation, m_random)] = run

    obs = pd.Series(run["r_obs_smoothed"].values, dtype=float)
    agent = pd.Series(run["r_agent_smoothed"].values, dtype=float)
    valid = pd.DataFrame({"obs": obs, "agent": agent}).dropna()
    mae = float((valid["obs"] - valid["agent"]).abs().mean()) if not valid.empty else float("inf")

    obs_drop_idx = int((valid["obs"] < 1.0).idxmax()) if (valid["obs"] < 1.0).any() else None
    agent_drop_idx = int((valid["agent"] < 1.0).idxmax()) if (valid["agent"] < 1.0).any() else None

    coarse_rows.append({
        "infectious_rate": selected_infectious_rate,
        "Mhome": m_home,
        "Moccupation": m_occupation,
        "Mrandom": m_random,
        "mae_r_obs_vs_agent": mae,
        "obs_drop_below_1_idx": obs_drop_idx,
        "agent_drop_below_1_idx": agent_drop_idx,
        "agent_drops_below_1": agent_drop_idx is not None,
        "timing_match_rough": (obs_drop_idx is not None and agent_drop_idx is not None and abs(agent_drop_idx - obs_drop_idx) <= 10),
    })

coarse_df = pd.DataFrame(coarse_rows).sort_values("mae_r_obs_vs_agent").reset_index(drop=True)
coarse_df


In [ ]:
refine_candidates = coarse_df.head(refine_top_n)[["Mhome", "Moccupation", "Mrandom"]].itertuples(index=False, name=None)
refine_rows = []
refine_runs = {}

for m_home, m_occupation, m_random in refine_candidates:
    weights = {
        "household": float(m_home),
        "work": float(m_occupation),
        "school": float(m_occupation),
        "random": float(m_random),
    }
    run = run_reff_calibration_scenario(
        region_config=region_config,
        observed_cases_path=str(observed_cases_path),
        timeline_processed_path=str(timeline_processed_path),
        mobility_processed_path=str(mobility_processed_path),
        reference_start_date=reference_start_date,
        end_date=end_date,
        initial_infected=initial_infected,
        simulated_population=simulated_population,
        infectious_rate=selected_infectious_rate,
        household_mobility_scale=m_home,
        work_mobility_scale=m_occupation,
        school_mobility_scale=m_occupation,
        random_mobility_scale=m_random,
        network_weights=weights,
        occupation_school_weight=occupation_school_weight,
        generation_interval=generation_interval,
        smoothing_window=smoothing_window,
        use_openabm=use_openabm,
    )
    refine_runs[(m_home, m_occupation, m_random)] = run

    obs = pd.Series(run["r_obs_smoothed"].values, dtype=float)
    agent = pd.Series(run["r_agent_smoothed"].values, dtype=float)
    valid = pd.DataFrame({"obs": obs, "agent": agent}).dropna()
    mae = float((valid["obs"] - valid["agent"]).abs().mean()) if not valid.empty else float("inf")

    obs_drop_idx = int((valid["obs"] < 1.0).idxmax()) if (valid["obs"] < 1.0).any() else None
    agent_drop_idx = int((valid["agent"] < 1.0).idxmax()) if (valid["agent"] < 1.0).any() else None

    refine_rows.append({
        "infectious_rate": selected_infectious_rate,
        "Mhome": m_home,
        "Moccupation": m_occupation,
        "Mrandom": m_random,
        "mae_r_obs_vs_agent": mae,
        "obs_drop_below_1_idx": obs_drop_idx,
        "agent_drop_below_1_idx": agent_drop_idx,
        "agent_drops_below_1": agent_drop_idx is not None,
        "timing_match_rough": (obs_drop_idx is not None and agent_drop_idx is not None and abs(agent_drop_idx - obs_drop_idx) <= 10),
    })

refine_df = pd.DataFrame(refine_rows).sort_values("mae_r_obs_vs_agent").reset_index(drop=True)
refine_df


In [ ]:
best = refine_df.iloc[0]
best_key = (float(best["Mhome"]), float(best["Moccupation"]), float(best["Mrandom"]))
best_run = refine_runs[best_key]

obs_ts = best_run["r_obs_smoothed"]
agent_ts = best_run["r_agent_smoothed"]
obs_dates = pd.to_datetime(obs_ts.times)
agent_dates = pd.to_datetime([pd.Timestamp(reference_start_date) + pd.Timedelta(days=int(t)) for t in agent_ts.times])

intervention_set = InterventionSet(best_run["interventions"])
r_linear_values = []
for t in range(len(agent_ts.values)):
    multipliers = compile_network_multipliers(intervention_set, t=t)
    m_house = float(multipliers.get("household", 1.0)) * float(best_key[0])
    m_work = float(multipliers.get("work", 1.0)) * float(best_key[1])
    m_school = float(multipliers.get("school", 1.0)) * float(best_key[1])
    m_occupation = (1.0 - float(occupation_school_weight)) * m_work + float(occupation_school_weight) * m_school
    m_random = float(multipliers.get("random", 1.0)) * float(best_key[2])
    r_linear_values.append(float(R0_target) * ((m_house + m_occupation + m_random) / 3.0))

r_linear_smoothed = pd.Series(r_linear_values, dtype=float).rolling(window=smoothing_window, min_periods=1).mean().to_numpy()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(obs_dates.to_numpy(), pd.Series(obs_ts.values, dtype=float).to_numpy(), label="R_obs (smoothed)", linewidth=2)
ax.plot(agent_dates.to_numpy(), pd.Series(agent_ts.values, dtype=float).to_numpy(), label="R_agent (smoothed)", linewidth=2)
ax.plot(agent_dates.to_numpy(), r_linear_smoothed, label="R_linear (3-network controls)", linewidth=2)
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5, label="R=1")
ax.axhline(float(R0_target), color="tab:gray", linestyle="--", linewidth=1.5, label=f"R0_target={R0_target}")
ax.set_title(f"Phase 2 best fit: Mhome={best_key[0]}, Moccupation={best_key[1]}, Mrandom={best_key[2]}")
ax.set_xlabel("Date")
ax.set_ylabel("R indicator (smoothed)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print("Phase 2 interpretation")
print(f"- Best infectious_rate from Phase 1: {selected_infectious_rate}")
print(f"- Best M controls: Mhome={best_key[0]}, Moccupation={best_key[1]}, Mrandom={best_key[2]}")
print(f"- MAE(R_obs, R_agent): {float(best['mae_r_obs_vs_agent']):.3f}")

if bool(best["timing_match_rough"]):
    print("- R_agent drop timing is roughly aligned with R_obs.")
else:
    print("- Main mismatch still includes drop timing; tune M controls and/or baseline infectious_rate.")

if bool(best["agent_drops_below_1"]):
    print("- R_agent goes below 1 in this setup.")
else:
    print("- R_agent does not clearly go below 1; intervention magnitude likely too weak.")
